In [ ]:
# Configure the repository root once so the same notebook works locally and on Kaggle.
import os
import sys
from pathlib import Path

_configured_root = os.environ.get("DATA_DIR")
if _configured_root:
    DATA_DIR = Path(_configured_root).expanduser().resolve()
else:
    DATA_DIR = next(
        (p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
         if (p / "README.md").is_file() and (p / "data").is_dir()),
        None,
    )
    if DATA_DIR is None:
        raise FileNotFoundError("Set DATA_DIR to the repository root before running this notebook.")
os.environ["DATA_DIR"] = str(DATA_DIR)
os.chdir(DATA_DIR)
if str(DATA_DIR) not in sys.path:
    sys.path.insert(0, str(DATA_DIR))


In [ ]:
import time
import os
import json
import sys
import numpy as np
import pandas as pd
import torch
from tqdm import tqdm

# Thiết lập thiết bị tính toán
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Sử dụng thiết bị: {DEVICE}")
print("[Info] PyTorch Version:", torch.__version__)
print("[Info] CUDA Khả dụng:", torch.cuda.is_available())

# Bổ sung đường dẫn mô-đun evaluator
sys.path.append(str(DATA_DIR / "src/eval"))
from evaluator import evaluate

if torch.cuda.is_available():
    torch.cuda.reset_peak_memory_stats()

Sử dụng thiết bị: cuda
[Info] PyTorch Version: 2.11.0+cu128
[Info] CUDA Khả dụng: True


In [ ]:
import hashlib

md5 = lambda p: hashlib.md5(open(p, "rb").read()).hexdigest()

# Đường dẫn tệp dữ liệu theo nguồn ngocccchannn
RAW_NPY       = str(DATA_DIR / "data/course_llm/1_embeddings/raw_course_text_embeddings.npy")
LLM_NPY       = str(DATA_DIR / "data/course_llm/1_embeddings/llm_profile_embeddings.npy")
PROFILES_JSON = str(DATA_DIR / "data/course_llm/2_profiles_full_and_shards/full_3769_profiles.json")
COLD_TEST_JSON= str(DATA_DIR / "data/splits/cold_scenario/cold_items_test.json")
TRAIN_PARQUET = str(DATA_DIR / "data/splits/cold_scenario/train_cold.parquet")
TEST_PARQUET  = str(DATA_DIR / "data/splits/cold_scenario/test_cold.parquet")
VAL_PARQUET = str(DATA_DIR / "data/splits/cold_scenario/val_cold.parquet")

# Kiểm tra mã MD5 Checksum các tệp embedding đầu vào
print("--- MD5 Checksums ---")
for name, path in [("raw_emb", RAW_NPY), ("llm_emb", LLM_NPY)]:
    print(f"  {name:10s}: {md5(path)}")

# 1. Nạp ma trận embedding raw (384d) và llm (384d)
S_raw = np.load(RAW_NPY).astype(np.float32)
S_llm = np.load(LLM_NPY).astype(np.float32)

# Hàm chuẩn hóa L2 L2-norm
nz = lambda X: X / np.linalg.norm(X, axis=1, keepdims=True).clip(1e-9)

# Chuẩn bị ma trận đặc trưng cho 3 biến thể Content-KNN
S_raw_norm = nz(S_raw)                                        # Biến thể 1: Văn bản gốc (384 chiều)
S_llm_norm = nz(S_llm)                                        # Biến thể 2: Hồ sơ LLM (384 chiều)
S_fused    = np.concatenate([S_raw_norm, S_llm_norm], axis=1) # Biến thể 3: Gốc + LLM (768 chiều)
#S_fused    = nz(nz(S_raw) + nz(S_llm))      # Biến thể 3: Gốc _ LLM (384 chiều)

assert S_raw_norm.shape == (3769, 384) and S_fused.shape == (3769, 768), "Sai kích thước ma trận đặc trưng!"
print(f"\n[Xác nhận] Kích thước 3 biến thể đặc trưng: Raw={S_raw_norm.shape}, LLM={S_llm_norm.shape}, Fused={S_fused.shape}")

# Xây dựng bảng ánh xạ course_id từ full_3769_profiles.json
with open(PROFILES_JSON, 'r') as f:
    full_profiles = json.load(f)

id_to_index = {}
index_to_id = {}
for idx, profile in enumerate(full_profiles):
    cid = profile.get('course_id')
    if cid:
        id_to_index[str(cid)] = idx
        index_to_id[idx] = str(cid)

assert len(id_to_index) == 3769, "Số lượng khóa học trong bảng chỉ số không đúng 3.769!"

# 3. Khởi tạo mặt nạ tập ứng viên cand_a (502 khóa lạnh) và cand_b (3.769 khóa)
with open(COLD_TEST_JSON, 'r') as f:
    cold_items_test = json.load(f)

cand_bool_a = np.zeros(3769, dtype=bool)
for cid in cold_items_test:
    if str(cid) in id_to_index:
        cand_bool_a[id_to_index[str(cid)]] = True

cold_cand_tensor_a = torch.tensor(cand_bool_a, dtype=torch.bool)
cold_cand_tensor_b = torch.ones(3769, dtype=torch.bool)

print(f"Số lượng khóa lạnh tập (a): {cold_cand_tensor_a.sum().item()} / 502")
print(f"Số lượng khóa ứng viên tập (b): {cold_cand_tensor_b.sum().item()} / 3769")

--- MD5 Checksums ---
  raw_emb   : f80445649ef8a8217d048bd3e7ac0ad5
  llm_emb   : fc1819307be227a1544e1be0a4017393

[Xác nhận] Kích thước 3 biến thể đặc trưng: Raw=(3769, 384), LLM=(3769, 384), Fused=(3769, 768)
Số lượng khóa lạnh tập (a): 502 / 502
Số lượng khóa ứng viên tập (b): 3769 / 3769


In [4]:
# Nạp dữ liệu tương tác
tr_df = pd.read_parquet(TRAIN_PARQUET)
te_df = pd.read_parquet(TEST_PARQUET)

# Ánh xạ user_id đồng nhất
raw_target_dict = te_df.groupby('user_id')['course_id'].apply(list).to_dict()
unique_users = sorted(raw_target_dict.keys())
user_to_int = {u: i for i, u in enumerate(unique_users)}
int_to_user = {i: u for i, u in enumerate(unique_users)}

# Từ điển Ground-Truth cho tập Test (3.340 học viên)
target_dict = {
    user_to_int[u]: [id_to_index[str(cid)] for cid in gt if str(cid) in id_to_index] 
    for u, gt in raw_target_dict.items() if u in user_to_int
}

# Từ điển Mask che lịch sử tương tác trong Train
raw_mask_dict = tr_df.groupby('user_id')['course_id'].apply(list).to_dict()
mask_dict = {
    user_to_int[u]: [id_to_index[str(cid)] for cid in hist if str(cid) in id_to_index] 
    for u, hist in raw_mask_dict.items() if u in user_to_int
}

user_history = raw_mask_dict
print(f"Tổng số học viên cần đánh giá trên test: {len(target_dict)}")

Tổng số học viên cần đánh giá trên test: 3340


In [5]:
def build_user_embedding_matrix(embeddings_matrix, user_history_dict, c2i):
    """
    Biểu diễn học viên (u) = trung bình cộng vector L2-normalized của các khóa học 
    mà học viên đó đã tương tác trong tập huấn luyện (train_cold).
    """
    user_embs = {}
    for u, hist in user_history_dict.items():
        v_idxs = [c2i[str(cid)] for cid in hist if str(cid) in c2i]
        if len(v_idxs) > 0:
            u_emb = embeddings_matrix[v_idxs].mean(axis=0)
            norm = np.linalg.norm(u_emb)
            if norm > 0:
                u_emb = u_emb / norm
            user_embs[u] = u_emb
        else:
            user_embs[u] = np.zeros(embeddings_matrix.shape[1])
    return user_embs

from evaluator import topk_random_ties

In [6]:
@torch.no_grad()
def eval_subgroups(score_fn, targets, mask, cand, groups, ks=(5, 10, 20), tie_seed=0, bs=1024):
    """
    X xếp hạng trên TOÀN BỘ tập ứng viên cand (502 khóa cold-test). Với mỗi nhóm, chỉ số chỉ
    tính trên các đích thuộc nhóm, và chỉ cho học viên có ít nhất một đích trong nhóm.
    Nguồn: Phụ lục A.1 (Review-2026-10-22)
    """
    users = sorted(targets); K = max(ks)
    g = torch.Generator().manual_seed(tie_seed)
    disc = 1.0 / np.log2(np.arange(2, K + 2))
    cand_t = torch.as_tensor(cand) if cand is not None else None
    
    out = {n: {f"{m}@{k}": [] for m in ("Recall", "NDCG", "HR") for k in ks} for n in groups}
    
    for s in range(0, len(users), bs):
        ub = users[s:s + bs]
        sc = score_fn(torch.tensor(ub)).float().cpu().clone()
        if cand_t is not None:
            sc[:, ~cand_t] = -float("inf")
        for r, u in enumerate(ub):
            if mask.get(u):
                sc[r, list(mask[u])] = -float("inf")
        
        top = topk_random_ties(sc, K, g)
        
        for r, u in enumerate(ub):
            for n, gset in groups.items():
                gt = [i for i in targets[u] if i in gset]
                if not gt:
                    continue
                hit = np.isin(top[r], gt).astype(float)
                for k in ks:
                    h = hit[:k]
                    idcg = disc[:min(len(gt), k)].sum()
                    out[n][f"Recall@{k}"].append(h.sum() / len(gt))
                    out[n][f"NDCG@{k}"].append((h * disc[:k]).sum() / idcg)
                    out[n][f"HR@{k}"].append(float(h.sum() > 0))
                    
    return {n: {m: np.array(v) for m, v in d.items()} for n, d in out.items()}

In [7]:
@torch.no_grad()
def latency_ms_per_user(user_vecs_np, item_vecs_np, cand_mask, K=20, bs=1024, reps=5):
    """
    Đo độ trễ suy luận chuẩn trên GPU theo Phụ lục A.6.
    Chỉ đo phần chấm điểm + top-K trên GPU.
    """
    if not torch.cuda.is_available():
        return 0.0, 0.0
        
    U_t = torch.tensor(user_vecs_np, device=DEVICE, dtype=torch.float32)
    V_t = torch.tensor(item_vecs_np, device=DEVICE, dtype=torch.float32)
    cand_t = torch.as_tensor(cand_mask, device=DEVICE)
    
    def run():
        for s in range(0, len(U_t), bs):
            sc = U_t[s:s + bs] @ V_t.t()
            sc[:, ~cand_t] = -float("inf")
            sc.topk(K, dim=1)
            
    # Khởi động GPU (warmup)
    run(); run()
    
    times = []
    for _ in range(reps):
        torch.cuda.synchronize()
        t0 = time.perf_counter()
        run()
        torch.cuda.synchronize()
        times.append((time.perf_counter() - t0) * 1000 / len(U_t))
        
    return float(np.mean(times)), float(np.std(times, ddof=1))

In [8]:
# Tạo thư mục lưu điểm học viên
os.makedirs("results_knn", exist_ok=True)

VARIANTS = {
    "raw": ("Content-KNN (Văn bản gốc)", S_raw_norm),
    "llm": ("Content-KNN (Hồ sơ LLM)", S_llm_norm),
    "fused": ("Content-KNN (Gốc + LLM)", S_fused)
}

all_knn_results = {}

def create_score_fn(em_mat, u_dict):
    def score_fn(ub_tensor):
        indices = ub_tensor.tolist() if isinstance(ub_tensor, torch.Tensor) else list(ub_tensor)
        batch_users = [int_to_user[int(idx)] for idx in indices]
        batch_vecs = np.array([u_dict.get(u, np.zeros(em_mat.shape[1])) for u in batch_users])
        scores = np.dot(batch_vecs, em_mat.T)
        return torch.tensor(scores, dtype=torch.float32)
    return score_fn

for key, (name, emb_matrix) in VARIANTS.items():
    print(f"\n================ ĐANG CHẠY ĐÁNH GIÁ: {name} ================")
    
    norms = np.linalg.norm(emb_matrix, axis=1, keepdims=True)
    norms[norms == 0] = 1.0
    norm_embeddings = emb_matrix / norms
    
    user_embs_dict = build_user_embedding_matrix(norm_embeddings, user_history, id_to_index)
    
    score_fn = create_score_fn(norm_embeddings, user_embs_dict)
        
    # 1. Đánh giá Tập (a) - Khóa lạnh
    res_a = evaluate(score_fn, target_dict, mask_dict, ks=(5, 10, 20), cand=cold_cand_tensor_a, bs=1024, tie_seed=0)
    
    # 2. Đánh giá Tập (b) - Toàn bộ 3.769 khóa
    res_b = evaluate(score_fn, target_dict, mask_dict, ks=(5, 10, 20), cand=cold_cand_tensor_b, bs=1024, tie_seed=0)
    
    # 3. Lưu điểm NDCG@20 từng học viên (3.340 học viên) cho kiểm định V6
    np.save(f"results_knn/user_ndcg20_knn_{key}.npy", res_a["NDCG@20"])
    
    # 4. Đo độ trễ suy luận trên GPU (P-RQ4)
    user_vecs_array = np.array([user_embs_dict.get(int_to_user[i], np.zeros(emb_matrix.shape[1])) for i in sorted(target_dict.keys())])
    lat_mean, lat_std = latency_ms_per_user(user_vecs_array, emb_matrix, cold_cand_tensor_a)
    
    # BÁO CÁO CHI TIẾT TẤT CẢ CHỈ SỐ
    print(f"\n--- BÁO CÁO TỔNG HỢP CHI TIẾT ({name}) ---")
    print(f"Độ trễ GPU (P-RQ4): {lat_mean:.4f} ± {lat_std:.4f} ms/user")
    print(f"Đã lưu tệp điểm: results_knn/user_ndcg20_knn_{key}.npy (Độ dài: {len(res_a['NDCG@20'])})")
    
    print("\n[TẬP (a) - 502 Khóa lạnh chính]:")
    for m in ["Recall", "NDCG", "HR"]:
        metrics_str = " | ".join([f"{m}@{k}: {res_a[f'{m}@{k}'].mean():.4f}" for k in [5, 10, 20]])
        print(f"{metrics_str}")
        
    print("\n[TẬP (b) - Toàn bộ 3.769 khóa]:")
    for m in ["Recall", "NDCG", "HR"]:
        metrics_str = " | ".join([f"{m}@{k}: {res_b[f'{m}@{k}'].mean():.4f}" for k in [5, 10, 20]])
        print(f"{metrics_str}")
    
    all_knn_results[key] = {"res_a": res_a, "res_b": res_b, "score_fn": score_fn}


================ ĐANG CHẠY ĐÁNH GIÁ: Content-KNN (Văn bản gốc) ================

--- BÁO CÁO TỔNG HỢP CHI TIẾT (Content-KNN (Văn bản gốc)) ---
Độ trễ GPU (P-RQ4): 0.0024 ± 0.0000 ms/user
Đã lưu tệp điểm: results_knn/user_ndcg20_knn_raw.npy (Độ dài: 3340)

[TẬP (a) - 502 Khóa lạnh chính]:
Recall@5: 0.1157 | Recall@10: 0.1557 | Recall@20: 0.2002
NDCG@5: 0.0795 | NDCG@10: 0.0932 | NDCG@20: 0.1053
HR@5: 0.1243 | HR@10: 0.1749 | HR@20: 0.2311

[TẬP (b) - Toàn bộ 3.769 khóa]:
Recall@5: 0.0307 | Recall@10: 0.0715 | Recall@20: 0.0948
NDCG@5: 0.0200 | NDCG@10: 0.0326 | NDCG@20: 0.0386
HR@5: 0.0326 | HR@10: 0.0754 | HR@20: 0.0991

================ ĐANG CHẠY ĐÁNH GIÁ: Content-KNN (Hồ sơ LLM) ================

--- BÁO CÁO TỔNG HỢP CHI TIẾT (Content-KNN (Hồ sơ LLM)) ---
Độ trễ GPU (P-RQ4): 0.0024 ± 0.0000 ms/user
Đã lưu tệp điểm: results_knn/user_ndcg20_knn_llm.npy (Độ dài: 3340)

[TẬP (a) - 502 Khóa lạnh chính]:
Recall@5: 0.0716 | Recall@10: 0.1338 | Recall@20: 0.1808
NDCG@5: 0.0495 | NDCG@10: 0.

Đánh giá P-RQ3 phân nhóm khóa

In [9]:
# 1. Rút trích dup_group_id từ dữ liệu full_profiles và ánh xạ nhóm khóa 
dup_group_list = [profile.get('dup_group_id') for profile in full_profiles]
_, dup_int = np.unique(dup_group_list, return_inverse=True)

# 2. Xác định các khóa ấm có tương tác trong train_cold
warm_items_np = np.unique([id_to_index[str(c)] for c in tr_df["course_id"] if str(c) in id_to_index])
warm_dup_groups = set(dup_int[warm_items_np].tolist())

# 3. Tách danh sách cold_items_test thành nhóm trùng (dup) và mới (new)
cold_test_dup_ids = [c for c in cold_items_test if str(c) in id_to_index and dup_int[id_to_index[str(c)]] in warm_dup_groups]
cold_test_new_ids = [c for c in cold_items_test if str(c) in id_to_index and dup_int[id_to_index[str(c)]] not in warm_dup_groups]

# Kiểm tra số lượng bắt buộc (124 khóa trùng / 378 khóa mới)
assert len(cold_test_dup_ids) == 124 and len(cold_test_new_ids) == 378, \
    f"Sai số lượng khóa ứng viên! Trùng: {len(cold_test_dup_ids)}, Mới: {len(cold_test_new_ids)}"
print(f"[Xác nhận] Phân tách thành công P-RQ3 Test Split: {len(cold_test_dup_ids)} khóa trùng | {len(cold_test_new_ids)} khóa mới hoàn toàn.")

# 4. Xây dựng từ điển groups chứa index nguyên thủy cho hàm eval_subgroups
groups = {
    "trung": {id_to_index[str(c)] for c in cold_test_dup_ids if str(c) in id_to_index},
    "moi": {id_to_index[str(c)] for c in cold_test_new_ids if str(c) in id_to_index}
}

[Xác nhận] Phân tách thành công P-RQ3 Test Split: 124 khóa trùng | 378 khóa mới hoàn toàn.


In [10]:
# Nạp dữ liệu val_cold để xây dựng mask_for_test chuẩn theo data.py
va_df = pd.read_parquet(VAL_PARQUET)

# Xây dựng ground-truth cho val
val_targets = {}
for u_str, grp in va_df.groupby(va_df["user_id"].astype(str)):
    if u_str in user_to_int:
        val_targets[user_to_int[u_str]] = [id_to_index[str(cid)] for cid in grp["course_id"] if str(cid) in id_to_index]

# Mask chuẩn khi đánh giá test: che lịch sử train và val
mask_for_test = {u: list(items) for u, items in mask_dict.items()}
for u, v_items in val_targets.items():
    mask_for_test.setdefault(u, []).extend(v_items)

In [11]:
print(f"\n================ ĐÁNH GIÁ P-RQ3 PHÂN NHÓM KHÓA ================")

# Hàm hỗ trợ in đầy đủ Recall, NDCG, HR cho từng nhóm
def print_subgroup_metrics(subgroup_res, model_name):
    print(f"\n--- CHI TIẾT P-RQ3 CHO: {model_name} ---")
    for subgroup, metrics in subgroup_res.items():
        print(f"\n  [Nhóm: {subgroup.upper()}] (Số lượng học viên: {len(metrics['Recall@20'])})")
        for m in ["Recall", "NDCG", "HR"]:
            metrics_str = " | ".join([f"{m}@{k}: {metrics[f'{m}@{k}'].mean():.4f}" for k in [5, 10, 20]])
            print(f"{metrics_str}")

# 5. Chạy P-RQ3 cho Content-KNN (Văn bản gốc) và Kiểm thử bắt buộc 
res_knn_raw_sub = eval_subgroups(
    score_fn=all_knn_results["raw"]["score_fn"],
    targets=target_dict,
    mask=mask_for_test,
    cand=cold_cand_tensor_a,
    groups=groups,
    ks=(5, 10, 20),
    tie_seed=0
)

# Assertion Test bắt buộc theo Phụ lục A.1[cite: 13] và Mục 2.3[cite: 4]
assert len(res_knn_raw_sub["trung"]["Recall@20"]) == 1028 and len(res_knn_raw_sub["moi"]["Recall@20"]) == 2844, \
    f"Lỗi: Số lượng học viên không khớp! (Trùng={len(res_knn_raw_sub['trung']['Recall@20'])}, Mới={len(res_knn_raw_sub['moi']['Recall@20'])})"
assert abs(res_knn_raw_sub["trung"]["Recall@20"].mean() - 0.2642) < 1e-3, \
    f"Lỗi: Recall@20 nhóm trùng không đạt 0.2642! (Thực tế: {res_knn_raw_sub['trung']['Recall@20'].mean():.4f})"
assert abs(res_knn_raw_sub["moi"]["Recall@20"].mean() - 0.1595) < 1e-3, \
    f"Lỗi: Recall@20 nhóm mới không đạt 0.1595! (Thực tế: {res_knn_raw_sub['moi']['Recall@20'].mean():.4f})"

print("KIỂM THỬ BẮT BUỘC P-RQ3 CHO CONTENT-KNN (VĂN BẢN GỐC) THÀNH CÔNG!")
print_subgroup_metrics(res_knn_raw_sub, VARIANTS["raw"][0])

# 6. Đánh giá và in chi tiết P-RQ3 cho 2 biến thể còn lại (Hồ sơ LLM và Gốc + LLM)
for key in ["llm", "fused"]:
    res_sub = eval_subgroups(
        score_fn=all_knn_results[key]["score_fn"],
        targets=target_dict,
        mask=mask_for_test,
        cand=cold_cand_tensor_a,
        groups=groups,
        ks=(5, 10, 20),
        tie_seed=0
    )
    print_subgroup_metrics(res_sub, VARIANTS[key][0])


================ ĐÁNH GIÁ P-RQ3 PHÂN NHÓM KHÓA ================
KIỂM THỬ BẮT BUỘC P-RQ3 CHO CONTENT-KNN (VĂN BẢN GỐC) THÀNH CÔNG!

--- CHI TIẾT P-RQ3 CHO: Content-KNN (Văn bản gốc) ---

  [Nhóm: TRUNG] (Số lượng học viên: 1028)
Recall@5: 0.1213 | Recall@10: 0.1865 | Recall@20: 0.2642
NDCG@5: 0.0926 | NDCG@10: 0.1140 | NDCG@20: 0.1341
HR@5: 0.1235 | HR@10: 0.1936 | HR@20: 0.2772

  [Nhóm: MOI] (Số lượng học viên: 2844)
Recall@5: 0.0967 | Recall@10: 0.1261 | Recall@20: 0.1595
NDCG@5: 0.0623 | NDCG@10: 0.0722 | NDCG@20: 0.0810
HR@5: 0.1027 | HR@10: 0.1378 | HR@20: 0.1758

--- CHI TIẾT P-RQ3 CHO: Content-KNN (Hồ sơ LLM) ---

  [Nhóm: TRUNG] (Số lượng học viên: 1028)
Recall@5: 0.0696 | Recall@10: 0.1207 | Recall@20: 0.1905
NDCG@5: 0.0501 | NDCG@10: 0.0663 | NDCG@20: 0.0841
HR@5: 0.0730 | HR@10: 0.1235 | HR@20: 0.1946

  [Nhóm: MOI] (Số lượng học viên: 2844)
Recall@5: 0.0627 | Recall@10: 0.1186 | Recall@20: 0.1526
NDCG@5: 0.0422 | NDCG@10: 0.0602 | NDCG@20: 0.0691
HR@5: 0.0675 | HR@10: 0.12

In [12]:
os.makedirs("scores", exist_ok=True)

for key in ["raw", "llm", "fused"]:
    # Chạy lại eval_subgroups cho từng biến thể để lấy dictionary kết quả
    sub_res = eval_subgroups(
        score_fn=all_knn_results[key]["score_fn"],
        targets=target_dict,
        mask=mask_for_test,
        cand=cold_cand_tensor_a,
        groups=groups,
        ks=(20,),
        tie_seed=0
    )
    
    # Lưu mảng NDCG@20 cho từng nhóm
    for group_name in ["trung", "moi"]:
        ndcg_arr = sub_res[group_name]["NDCG@20"]
        file_path = f"scores/user_ndcg20_knn_{key}_{group_name}.npy"
        np.save(file_path, ndcg_arr)
        print(f"Đã lưu tệp phân nhóm Content-KNN: {file_path} (Kích thước: {ndcg_arr.shape})")

print("Hoàn tất chuẩn bị các tệp .npy phân nhóm P-RQ3 cho Content-KNN!")

Đã lưu tệp phân nhóm Content-KNN: scores/user_ndcg20_knn_raw_trung.npy (Kích thước: (1028,))
Đã lưu tệp phân nhóm Content-KNN: scores/user_ndcg20_knn_raw_moi.npy (Kích thước: (2844,))
Đã lưu tệp phân nhóm Content-KNN: scores/user_ndcg20_knn_llm_trung.npy (Kích thước: (1028,))
Đã lưu tệp phân nhóm Content-KNN: scores/user_ndcg20_knn_llm_moi.npy (Kích thước: (2844,))
Đã lưu tệp phân nhóm Content-KNN: scores/user_ndcg20_knn_fused_trung.npy (Kích thước: (1028,))
Đã lưu tệp phân nhóm Content-KNN: scores/user_ndcg20_knn_fused_moi.npy (Kích thước: (2844,))
Hoàn tất chuẩn bị các tệp .npy phân nhóm P-RQ3 cho Content-KNN!


In [ ]:
# Đường dẫn tới một file điểm cần kiểm tra
file_path = str(DATA_DIR / "scores/user_ndcg20_knn_raw_trung.npy")

if os.path.exists(file_path):
    data = np.load(file_path)
    print(f"--- THÔNG TIN TỆP: {file_path} ---")
    print(f"Kích thước (Shape): {data.shape}")  # Kỳ vọng mong đợi: (1028,) đối với nhóm trùng
    print(f"Kiểu dữ liệu (Dtype): {data.dtype}")
    print(f"Giá trị NDCG@20 trung bình của nhóm: {np.mean(data):.4f}")
    print(f"5 giá trị điểm NDCG@20 đầu tiên của học viên: {data[:5]}")
else:
    print(f"Không tìm thấy tệp tại đường dẫn: {file_path}. Vui lòng kiểm tra lại quá trình chạy cell lưu file.")